In [1]:
from huggingface_hub import hf_hub_download
hf_hub_download(repo_id="benmoseley/ese-dl-2024-25-group-project", filename="train.h5", repo_type="dataset", local_dir="data")
hf_hub_download(repo_id="benmoseley/ese-dl-2024-25-group-project", filename="events.csv", repo_type="dataset", local_dir="data")
import pandas as pd
import h5py
import IPython
import PIL
import matplotlib.pyplot as plt
import os
import io
# this DataFrame contains all of the event meta data
df = pd.read_csv("data/events.csv", parse_dates=["start_utc"])

# this function loads all of the image arrays for a given id
def load_event(id):
    "Load event"
    with h5py.File(f'data/train.h5','r') as f:
        event = {img_type: f[id][img_type][:] for img_type in ['vis', 'ir069', 'ir107', 'vil', 'lght']}
    return event

# import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 去重，确保每个事件只保留一行
events = df.drop_duplicates(subset=['id']).copy()

# 时间划分
events['start_date'] = pd.to_datetime(events['start_utc']).dt.date
events['month'] = pd.to_datetime(events['start_utc']).dt.month
events['season'] = events['month'].map({
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Fall', 10: 'Fall', 11: 'Fall'
})

# 地理区域划分
def assign_region(row):
    if row['llcrnrlat'] > 40:  # 纬度划分
        return 'North'
    elif row['llcrnrlat'] < 30:
        return 'South'
    elif row['llcrnrlon'] < -100:  # 经度划分
        return 'West'
    else:
        return 'East'

events['region'] = events.apply(assign_region, axis=1)

# 分层抽样划分
train_events, val_events = train_test_split(
    events,
    test_size=0.3,  # 验证集比例
    stratify=events[['season', 'region']],  # 按季节和区域分层
    random_state=42
)

val_events, test_events = train_test_split(
    val_events,
    test_size=0.5,  # 测试集比例
    stratify=val_events[['season', 'region']],  # 按季节和区域分层
    random_state=42
)

train_ids = train_events['id']
val_ids = val_events['id']
test_ids = test_events['id']
print(f"训练集大小: {len(train_ids)}")
print(f"验证集大小: {len(val_ids)}")
print(f"测试集大小: {len(test_ids)}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


训练集大小: 560
验证集大小: 120
测试集大小: 120


In [2]:

import torch
import torch.nn.functional as F
import numpy as np

def process_data(event_ids):

    # 预定义数据大小（假设 N 是已知的）
    N = len(event_ids)
    H, W = 384, 384
    input_channels = 3  # 3 通道的时间序列长度 36
    target_channels = 1  # vil 时间序列 36
    frames = 36
    # 直接分配 tensor 避免 list append
    data_inputs = torch.empty((N, frames, input_channels, H, W), dtype=torch.float32)
    data_targets = torch.empty((N, frames, target_channels, H, W), dtype=torch.float32)

    for i, id in enumerate(event_ids):
        event = load_event(id)

        # 读取数据并转换为 Tensor
        vis = torch.from_numpy(event['vis'].astype(np.float32)).permute(2, 0, 1).unsqueeze(0)  # (36, 384, 384)
        ir069 = torch.from_numpy(event['ir069'].astype(np.float32)).permute(2, 0, 1).unsqueeze(0)  # (36, 原始 H, W)
        ir107 = torch.from_numpy(event['ir107'].astype(np.float32)).permute(2, 0, 1).unsqueeze(0)  # (36, 原始 H, W)
        vil = torch.from_numpy(event['vil'].astype(np.float32)).permute(2, 0, 1).unsqueeze(1)  # (36, 384, 384)

        # 上采样红外数据到 384x384
        ir069 = F.interpolate(ir069, size=(H, W), mode='bilinear', align_corners=False)
        ir107 = F.interpolate(ir107, size=(H, W), mode='bilinear', align_corners=False)

        # 归一化

        def normalize(tensor):
            min_val = tensor.min(dim=1, keepdim=True)[0].min(dim=2, keepdim=True)[0]  # 只计算一次 min
            max_val = tensor.max(dim=1, keepdim=True)[0].max(dim=2, keepdim=True)[0]  # 只计算一次 max
            return (tensor - min_val) / (max_val - min_val + 1e-8)  # 避免除零错误

        vis = normalize(vis)
        ir069 = normalize(ir069)
        ir107 = normalize(ir107)
        vil = normalize(vil)

        # 合并输入通道 (3, 36, 384, 384)
        inputs = torch.cat([vis, ir069, ir107], dim=0).permute(1, 0, 2, 3)  # (36, 3, 384, 384)

        # 存入预分配张量
        data_inputs[i] = inputs
        data_targets[i] = vil

        if (i + 1) % 10 == 0:
            print(f"Processed {i+1}/{N}")
    data_inputs = data_inputs.view(N * frames, 3, 384, 384)
    data_targets = data_targets.view(N * frames, 1, 384, 384)
    return data_inputs, data_targets
train_inputs, train_targets = process_data(train_ids)
val_inputs, val_targets = process_data(val_ids)
test_inputs, test_targets = process_data(test_ids)
# 检查最终形状
print("Final Input Shape:", train_inputs.shape, "Final Target Shape:", train_targets.shape)
print("Final Input Shape:", val_inputs.shape, "Final Target Shape:", val_targets.shape)
print("Final Input Shape:", test_inputs.shape, "Final Target Shape:", test_targets.shape)

Processed 10/560
Processed 20/560
Processed 30/560
Processed 40/560
Processed 50/560
Processed 60/560
Processed 70/560
Processed 80/560
Processed 90/560
Processed 100/560
Processed 110/560
Processed 120/560
Processed 130/560
Processed 140/560
Processed 150/560
Processed 160/560
Processed 170/560
Processed 180/560
Processed 190/560
Processed 200/560
Processed 210/560
Processed 220/560
Processed 230/560
Processed 240/560
Processed 250/560
Processed 260/560
Processed 270/560
Processed 280/560
Processed 290/560
Processed 300/560
Processed 310/560
Processed 320/560
Processed 330/560
Processed 340/560
Processed 350/560
Processed 360/560
Processed 370/560
Processed 380/560
Processed 390/560
Processed 400/560
Processed 410/560
Processed 420/560
Processed 430/560
Processed 440/560
Processed 450/560
Processed 460/560
Processed 470/560
Processed 480/560
Processed 490/560
Processed 500/560
Processed 510/560
Processed 520/560
Processed 530/560
Processed 540/560
Processed 550/560
Processed 560/560
P